In [1]:
import pandas as pd
import numpy as np
import kagglehub

path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Dataset downloaded to: {path}")

orders = pd.read_csv(f"{path}/olist_orders_dataset.csv")
items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
payments = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{path}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
cat_translation = pd.read_csv(f"{path}/product_category_name_translation.csv")

print(f"Orders shape: {orders.shape}")
print(f"Items shape: {items.shape}")
print(f"Customers shape: {customers.shape}")
print(f"Payments shape: {payments.shape}")
print(f"Reviews shape: {reviews.shape}")
print(f"Products shape: {products.shape}")
print(f"Sellers shape: {sellers.shape}")

100%|█████████████████████████████████████████████████████████████████████████████| 42.6M/42.6M [00:17<00:00, 2.50MB/s]

Extracting files...


Dataset downloaded to: C:\Users\shado\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
Orders shape: (99441, 8)
Items shape: (112650, 7)
Customers shape: (99441, 5)
Payments shape: (103886, 5)
Reviews shape: (99224, 7)
Products shape: (32951, 9)
Sellers shape: (3095, 4)


In [2]:
def load_and_clean_olist_data(path):
    orders = pd.read_csv(f"{path}/olist_orders_dataset.csv")
    items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
    customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
    payments = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
    reviews = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
    products = pd.read_csv(f"{path}/olist_products_dataset.csv")
    sellers = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
    cat_translation = pd.read_csv(f"{path}/product_category_name_translation.csv")

    datetime_cols = [
        'order_purchase_timestamp', 'order_approved_at', 
        'order_delivered_carrier_date', 'order_delivered_customer_date', 
        'order_estimated_delivery_date'
    ]
    for col in datetime_cols:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')

    orders_cleaned = orders.dropna(subset=['order_delivered_customer_date'])

    products_translated = pd.merge(products, cat_translation, on='product_category_name', how='left')
    products_translated['product_category_name_english'] = products_translated['product_category_name_english'].fillna('other')
    products_translated.drop(columns=['product_category_name'], inplace=True)

    reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
    reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

    master_df = pd.merge(orders_cleaned, items, on='order_id', how='inner')
    master_df = pd.merge(master_df, customers, on='customer_id', how='left')
    master_df = pd.merge(master_df, payments, on='order_id', how='left')
    master_df = pd.merge(master_df, products_translated, on='product_id', how='left')
    master_df = pd.merge(master_df, sellers, on='seller_id', how='left')
    master_df = pd.merge(master_df, reviews, on='order_id', how='left')

    master_df['actual_delivery_days'] = (master_df['order_delivered_customer_date'] - master_df['order_purchase_timestamp']).dt.days
    master_df['estimated_delivery_days'] = (master_df['order_estimated_delivery_date'] - master_df['order_purchase_timestamp']).dt.days
    master_df['delivery_delay_days'] = (master_df['order_delivered_customer_date'] - master_df['order_estimated_delivery_date']).dt.days
    master_df['is_delayed'] = master_df['delivery_delay_days'].apply(lambda x: 1 if x > 0 else 0)

    master_df['purchase_year'] = master_df['order_purchase_timestamp'].dt.year
    master_df['purchase_month'] = master_df['order_purchase_timestamp'].dt.month
    master_df['purchase_day_name'] = master_df['order_purchase_timestamp'].dt.day_name()

    return master_df

master_cleaned = load_and_clean_olist_data(path)

In [3]:
print("Master Cleaned Dataset Shape:", master_cleaned.shape)

print("\n--- First 5 Rows of Master Dataset ---")
print(master_cleaned.head())

critical_cols = [
    'actual_delivery_days', 'delivery_delay_days', 
    'is_delayed', 'product_category_name_english', 
    'payment_type'
]
print("\n--- Missing Values in Critical Columns ---")
print(master_cleaned[critical_cols].isnull().sum())

master_cleaned.to_csv("olist_master_cleaned.csv", index=False)
print("\nSuccess: Dataset is cleaned and saved to 'olist_master_cleaned.csv'")

Master Cleaned Dataset Shape: (115722, 46)

--- First 5 Rows of Master Dataset ---
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
3  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
4  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
2    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
3    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
4    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00     

In [7]:
df = pd.read_csv("olist_master_cleaned.csv")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df.sample(2).T)


                                                                          37103                             110941
order_id                                        ba32ca81e6f254f77b25b2959a1bb776  afef0360db6e6f6eb46b9600cd8c5450
customer_id                                     32b58a7d02e96a3bd310bb3dfa5baec9  61c245c5ac166a54e51ae198d45aaaaf
order_status                                                           delivered                         delivered
order_purchase_timestamp                                     2018-06-16 21:10:37               2017-10-27 16:34:10
order_approved_at                                            2018-06-19 04:16:56               2017-10-30 11:32:41
order_delivered_carrier_date                                 2018-06-20 14:58:00               2017-11-01 19:22:42
order_delivered_customer_date                                2018-06-21 19:56:54               2017-11-13 10:31:16
order_estimated_delivery_date                                         2018-06-29